# 08.2 — Writing .mat files for MATLAB

The whole point of porting to Python but keeping MATLAB parity is that the *analysis* stays in MATLAB. So a Python run must produce a `CM_Table.mat` — a results file MATLAB scripts load and plot *without modification* (Critical Note #16). This means writing a MATLAB **struct** from Python, with exactly the right field names, dtypes, and array shapes. `scipy.io.savemat` does the writing; the discipline is in getting the schema byte-compatible. This notebook is that results-file writer.

This notebook covers:

* Writing MATLAB structs from Python with `scipy.io.savemat`.
* The `CM_Table` schema: its six field types, dtypes, and shapes.
* MATLAB shape conventions (`oned_as="column"`, `single` vs `double`).
* The two result files: validation (per-epoch) and test (final).

**Prerequisites:** [08.1 folder hierarchy](08.1_folder_hierarchy_generation.ipynb), [06.6 confidence routing](../06_loss_orchestration/06.6_confidence_routing.ipynb).

## Section 1 — What MATLAB does

MATLAB builds a `CM_Table` (a `table` or struct of per-trial results) and `save`s it into the run's leaf folder ([08.1](08.1_folder_hierarchy_generation.ipynb)):

```matlab
CM_Table = cgg_generateBlankCMTable(numTrials, numDims);
CM_Table.TrueValue = targets;
CM_Table.Aggregation_Prediction = predictions;      % from cgg_getClassifierOutputsFromProbabilities
CM_Table.TrialConfidence = trialConf;
save(fullfile(foldDir, 'CM_Table.mat'), 'CM_Table');
```

The analysis scripts `DATA_cggAllNetworkEncoderResults.m` → `FIGURE_cggAllNetworkEncoderResults.m` later `load` this file and access `CM_Table.Aggregation_Prediction` etc. to compute accuracy and draw confusion matrices. The port must produce a `.mat` those scripts can read unchanged — `write_cm_table_mat` (Critical Note #16).

## Section 2 — The Python concepts you need

### 2.1 — Writing MATLAB structs from Python

`scipy.io.savemat(path, mdict)` writes a `.mat` file from a Python dict. Each key becomes a MATLAB variable; a nested dict becomes a MATLAB **struct** whose fields are the inner keys. So `{"CM_Table": {"TrueValue": arr, ...}}` lands in MATLAB as a struct `CM_Table` with a field `TrueValue`. NumPy arrays become MATLAB matrices. That's the entire bridge — but the *details* (dtype, shape, orientation) determine whether MATLAB reads it the way the analysis expects.

In [ ]:
import numpy as np, tempfile, os
from scipy.io import savemat, loadmat

# A minimal struct-of-arrays → MATLAB struct:
tmp = tempfile.mkdtemp()
p = os.path.join(tmp, "demo.mat")
savemat(p, {"MyStruct": {"values": np.array([1.0, 2.0, 3.0]), "label": "hello"}}, oned_as="column")

back = loadmat(p)["MyStruct"]
print("MATLAB struct fields:", back.dtype.names)
print("values shape:", back["values"][0, 0].shape, "→ column vector (oned_as='column')")
print("label:", back["label"][0, 0])

### 2.2 — The `CM_Table` schema

`write_cm_table_mat` builds the results struct with six kinds of field, one row per trial:

| Field | dtype | shape | meaning |
|---|---|---|---|
| `DataNumber` | `single` (float32) | `(N, 1)` | trial id (global, 1-indexed for MATLAB) |
| `TrueValue` | `double` | `(N, D)` | ground-truth class per dimension |
| `Window_1 … Window_K` | `double` | `(N, D)` | per-window prediction (one field each) |
| `Aggregation_Prediction` | `double` | `(N, D)` | cross-window aggregate ([06.5](../06_loss_orchestration/06.5_mil_softmax_pooling.ipynb)) |
| `TrialConfidence` | `double` | `(N, 1)` | per-trial confidence ([06.6](../06_loss_orchestration/06.6_confidence_routing.ipynb)) |
| `TaskConfidence` | `double` | `(N, D)` | per-dimension confidence |

`N` = trials, `D` = output dimensions, `K` = analysis windows. Write one and read it back:

In [ ]:
from neural_data_decoding.interop.cm_table_format import write_cm_table_mat

N, D = 5, 3
rng = np.random.default_rng(0)
path = os.path.join(tmp, "CM_Table.mat")
write_cm_table_mat(path,
    data_numbers=np.arange(N) + 1,                       # 1-indexed trial ids
    true_values=rng.integers(0, 4, (N, D)).astype(float),
    window_predictions=[rng.integers(0, 4, (N, D)).astype(float)],   # one window
    aggregation_prediction=None)                          # single window → defaults to it

tbl = loadmat(path)["CM_Table"]
for f in tbl.dtype.names:
    cell = tbl[f][0, 0]
    print(f"  {f:24} shape {str(cell.shape):8} dtype {cell.dtype}")

### 2.3 — Shape and dtype conventions matter

Three `savemat` details make the file MATLAB-compatible:

* **`oned_as="column"`** — a 1-D NumPy array would be ambiguous (row or column?); MATLAB is column-major and the analysis expects column vectors, so 1-D arrays are written as `(N, 1)`. Without this, `DataNumber` and `TrialConfidence` would come back as rows and break indexing.
* **`single` for `DataNumber`, `double` for everything else** — MATLAB distinguishes `single` (float32) and `double` (float64). The reference tables store `DataNumber` as `single`; matching it avoids type-mismatch surprises in the analysis. (Note: `DataNumber` is a *global, sparse* trial id — not `1..N` — so it's stored as a float, not an int index.)
* **`do_compression=True`** — compresses the `.mat`, matching the pipeline's output size and load behavior.

These aren't cosmetic: the analysis scripts assume them, and a wrong orientation or dtype produces subtly wrong plots (or errors) rather than a clean failure — the same silent-parity hazard as [08.1 §2.5](08.1_folder_hierarchy_generation.ipynb).

### 2.4 — TrialConfidence vs TaskConfidence: the shape asymmetry

In [ ]:
# TrialConfidence is per-TRIAL (N,1); TaskConfidence is per-DIMENSION (N,D).
print("TrialConfidence shape:", tbl["TrialConfidence"][0, 0].shape, "→ one scalar per trial")
print("TaskConfidence  shape:", tbl["TaskConfidence"][0, 0].shape, "→ one value per (trial, dimension)")
print()
print("both defaulted to ones (confidence heads disabled in this write):")
print("  TrialConfidence[:3]:", tbl["TrialConfidence"][0, 0][:3, 0])
print("  TaskConfidence[0]:  ", tbl["TaskConfidence"][0, 0][0])

The two confidence fields have *different* shapes because they measure different things ([06.6 §2.2](../06_loss_orchestration/06.6_confidence_routing.ipynb)): trial confidence is one number for the whole trial (`(N, 1)`), while task confidence is per-dimension (`(N, D)`) — is *this* decoding target certain? When the confidence heads are disabled, both fields are filled with **ones** (maximally confident), so the analysis scripts always find the fields with valid shapes even for a non-confidence run. The writer handles this defaulting so callers don't special-case it.

### 2.5 — Two files: validation (per-epoch) and test (final)

In [ ]:
from neural_data_decoding.interop.cm_table_format import (
    VALIDATION_CM_TABLE_FILENAME, TEST_CM_TABLE_FILENAME)
print("written EACH epoch (for model selection):", VALIDATION_CM_TABLE_FILENAME)
print("written ONCE at the end (final results):  ", TEST_CM_TABLE_FILENAME)
print()
print("→ Validation table: computed on the val set whenever a new best appears (05.5's Optimal).")
print("  Test table: computed once at the end from the RESTORED Optimal weights on the test set.")

The run produces two result tables, mirroring the checkpoint state machine ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)):

* **`CM_Table_Validation.mat`** — rewritten whenever a new best-validation epoch appears (the "Optimal" snapshot). MATLAB's `cgg_saveValidationCMTable.m` uses it for model selection.
* **`CM_Table.mat`** — written *once*, at the end, by running the *restored Optimal weights* on the held-out *test* set. This is the number the paper reports; the aggregator ([08.6](08.6_running_matlab_analysis_on_python_output.ipynb)) collects these.

Both land in the leaf `Fold_{N}/` directory ([08.1 §2.3](08.1_folder_hierarchy_generation.ipynb)). Writing validation-during-training and test-at-the-end from Optimal weights is exactly MATLAB's `cgg_trainNetwork.m` behavior (both saves gated on `IsOptimal`).

## Section 3 — The `neural_data_decoding` implementation

`write_cm_table_mat` — the signature and the `savemat` call:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/interop/cm_table_format.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def write_cm_table_mat"))
for k in range(i, i + 10):
    print(f"{k + 1:4} | {src[k]}")
print("...")
j0 = next(j for j, line in enumerate(src) if "savemat(" in line)
for k in range(j0, j0 + 5):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The keyword-only signature: `data_numbers`, `true_values`, `window_predictions` (a *list*, one array per window), plus optional `aggregation_prediction`, `trial_confidence`, `task_confidence`.
* `savemat(str(output_path), {"CM_Table": table}, do_compression=True, oned_as="column")` — the single serialization call (§2.3). The outer key `"CM_Table"` is the MATLAB variable name the analysis `load`s.
* When more than one window is supplied, `aggregation_prediction` is **required** (there's no single obvious aggregate); with one window it defaults to that window ([06.5](../06_loss_orchestration/06.5_mil_softmax_pooling.ipynb)).
* Confidence fields default to **ones** when not passed (§2.4) — the struct always has all fields with valid shapes.
* Critical Note #16: this is the primary MATLAB-interop surface — the schema is pinned against a real fixture (`tests/fixtures/reference_cm_tables/CM_Table.mat`, 106 trials × 4 dims × 59 windows) by the parity tests ([08.4](08.4_the_mat_round_trip_test.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — the round-trip preserves everything

Write a table with known values, load it back, and confirm the values survive exactly (the basis of the [08.4](08.4_the_mat_round_trip_test.ipynb) parity test).

In [ ]:
# Reveal:
true = np.array([[0, 1], [2, 3], [1, 0]], dtype=float)
pred = np.array([[0, 1], [2, 2], [1, 0]], dtype=float)
p2 = os.path.join(tmp, "rt.mat")
write_cm_table_mat(p2, data_numbers=np.array([10, 20, 30]),
                   true_values=true, window_predictions=[pred], aggregation_prediction=None)
loaded = loadmat(p2)["CM_Table"]
print("TrueValue survived exactly?", np.array_equal(loaded["TrueValue"][0, 0], true))
print("DataNumber (1-indexed ids):", loaded["DataNumber"][0, 0].ravel())
print("→ savemat/loadmat is a lossless round-trip for the schema.")

### Exercise 4.2 — multi-window needs an explicit aggregate

Show that supplying two windows *without* `aggregation_prediction` raises (there's no unambiguous aggregate), while supplying it produces `Window_1`, `Window_2`, and `Aggregation_Prediction`.

In [ ]:
# Reveal:
w1, w2 = np.zeros((3, 2)), np.ones((3, 2))
try:
    write_cm_table_mat(os.path.join(tmp, "mw.mat"), data_numbers=np.arange(3) + 1,
        true_values=np.zeros((3, 2)), window_predictions=[w1, w2], aggregation_prediction=None)
except ValueError as e:
    print("two windows, no aggregate →", type(e).__name__, ":", str(e)[:55])

write_cm_table_mat(os.path.join(tmp, "mw.mat"), data_numbers=np.arange(3) + 1,
    true_values=np.zeros((3, 2)), window_predictions=[w1, w2], aggregation_prediction=np.full((3, 2), 0.5))
print("with aggregate → fields:", loadmat(os.path.join(tmp, "mw.mat"))["CM_Table"].dtype.names)

## Section 5 — Common errors

### MATLAB reads column vectors as rows (or vice versa)

`oned_as` wasn't set to `"column"` (§2.3), so 1-D arrays got the wrong orientation. `DataNumber`/`TrialConfidence` must be `(N, 1)`. The writer sets this; a hand-rolled `savemat` must too.

### `DataNumber` type mismatch in the analysis

It should be `single` (float32), not `int` or `double` (§2.3). The reference tables use `single`; the analysis may assume it. Cast accordingly (the writer does).

### The analysis can't find a field

A field name typo, or a confidence field omitted. All six field *kinds* must be present with valid shapes — the writer defaults confidence to ones (§2.4) so the struct is always complete. Check the field names match exactly (`Aggregation_Prediction`, not `AggregationPrediction`).

### Multi-window write raises

Expected without an explicit `aggregation_prediction` (§Exercise 4.2) — with more than one window there's no default aggregate. Pass the cross-window aggregate ([06.5](../06_loss_orchestration/06.5_mil_softmax_pooling.ipynb)).

### The test table is written from the wrong weights

`CM_Table.mat` must come from the *restored Optimal* weights on the *test* set (§2.5), not the final-epoch weights on validation. Writing it from Current weights reports the wrong (usually worse, possibly overfit) number. The lifecycle restores Optimal first ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)).

## Section 6 — Further reading

- [`src/neural_data_decoding/interop/cm_table_format.py`](../../src/neural_data_decoding/interop/cm_table_format.py) — `write_cm_table_mat` and the schema (Critical Note #16).
- [`scipy.io.savemat` docs](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.savemat.html) — the `oned_as`/`do_compression` options.
- [08.4 the .mat round-trip test](08.4_the_mat_round_trip_test.ipynb) — the parity gate that pins this schema.

Next up: **[08.3 monitor table compatibility](08.3_monitor_table_compatibility.ipynb)** — the per-epoch validation table and what MATLAB's monitor expects from it.